# Práctica 1: Aprendizaje Automático
**Predicción de Subscripsción a un Producto Bancario**

* **Autores:** Jaime del Campo Montaña & David Alexander Torres Paillacho
* **NIA's:** 100495957 100522187

In [25]:
%pip install pandas numpy matplotlib seaborn scikit-learn
#necesario para intalar la librería numpy, que es una dependencia de scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Jaime\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn
#necesario para instalr la libreía seaborn

In [2]:
import pandas as pd
import numpy as np

RANDOM_STATE = 100522187
np.random.seed(RANDOM_STATE)


In [3]:
dataframe = pd.read_pickle('bank_15.pkl')
#muestro las primeras filas del dataframe --> comprobación
display(dataframe.head())

#flas, columnas, tipos de datos y valores nulos
print("\nInformación del dataframe")
dataframe.info()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes



Información del dataframe
<class 'pandas.DataFrame'>
Index: 11000 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11000 non-null  int64 
 1   job        10920 non-null  object
 2   marital    11000 non-null  object
 3   education  10744 non-null  object
 4   default    11000 non-null  object
 5   balance    11000 non-null  int64 
 6   housing    11000 non-null  object
 7   loan       11000 non-null  object
 8   contact    11000 non-null  object
 9   day        11000 non-null  int64 
 10  month      11000 non-null  object
 11  duration   11000 non-null  int64 
 12  campaign   11000 non-null  int64 
 13  pdays      11000 non-null  int64 
 14  previous   11000 non-null  int64 
 15  poutcome   11000 non-null  object
 16  deposit    11000 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.5+ MB


A continuación analizaremos los datos del dataset


In [5]:
#número de instancias y variables
print(f"número de instancias (filas): {dataframe.shape[0]}")
print(f"número de variables (columnas): {dataframe.shape[1]}")
#comprobación de valores nulos
print("\nvalores null por columna")
nulos = dataframe.isnull().sum()
print(nulos[nulos > 0])

número de instancias (filas): 11000
número de variables (columnas): 17

valores null por columna
job           80
education    256
dtype: int64


In [6]:
#variables categóricas > 10
categorical_columans = dataframe.select_dtypes(include=['object']).columns
print("\nCardinalidad de variables categóricas")
for columna in categorical_columans:
    val_unicos = dataframe[columna].nunique()
    if val_unicos > 10:
        print(f"'{columna}'tiene alta cardinalidad: {val_unicos} valores únicos")
    else:
        print(f"'{columna}':{val_unicos} valores únicos")

#columnas constantes
print("\ncolumnas constantes")
constant_columnas = [columna for columna in dataframe.columns if dataframe[columna].nunique() <= 1]
if len(constant_columnas) == 0:
    print("no hay columnas constantes.")
else:
    print(f"columnas constantes encontradas: {constant_columnas}")


Cardinalidad de variables categóricas
'job'tiene alta cardinalidad: 12 valores únicos
'marital':3 valores únicos
'education':4 valores únicos
'default':2 valores únicos
'housing':2 valores únicos
'loan':2 valores únicos
'contact':3 valores únicos
'month'tiene alta cardinalidad: 12 valores únicos
'poutcome':4 valores únicos
'deposit':2 valores únicos

columnas constantes
no hay columnas constantes.


In [7]:
#comprobar desbalanceo de la variable objetivo
print("Distribución de la variable objetivo (deposit):")
print(dataframe['deposit'].value_counts(normalize=True))

#análisis de pdays
print("\nEstadísticas de pdays:")
print(dataframe['pdays'].describe())


Distribución de la variable objetivo (deposit):
deposit
no     0.525455
yes    0.474545
Name: proportion, dtype: float64

Estadísticas de pdays:
count    11000.000000
mean        51.308636
std        108.782842
min         -1.000000
25%         -1.000000
50%         -1.000000
75%         20.250000
max        854.000000
Name: pdays, dtype: float64


In [8]:
from sklearn.model_selection import train_test_split

#separar X e y
X = dataframe.drop('deposit', axis=1)
y = dataframe['deposit']

#división Holdout (2/3 train, 1/3 test) según instrucciones
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.33, random_state=RANDOM_STATE, stratify=y
)

print(f"Tamaño entrenamiento: {X_train.shape[0]}")
print(f"Tamaño test: {X_test.shape[0]}")

Tamaño entrenamiento: 7370
Tamaño test: 3630


In [9]:
from sklearn.model_selection import StratifiedKFold

cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring_metric = 'roc_auc'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

#separamos columnas por tipo
#variables numéricas estándar 
num_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'previous']

#variables categóricas con nulos
cat_cols_con_nulos = ['job', 'education']

#resto
cat_cols = ['marital', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] 

#pdays
pdays_col = ['pdays']


#transformacion de pdays
def transform_pdays(X):
    #transform_pdays recibe un array 2D, pero nosotros queremos trabajar con un array 1D para hacer las transformaciones más fácilmente.    
   
    pdays_valores = np.array(X).flatten() #aseguramos que es un array 1D
    
    no_previo = (pdays_valores == -1).astype(int) #1 si no hay contacto previo, 0 si sí lo hubo
    dias_validos = np.where(pdays_valores == -1, 0, pdays_valores) #reemplazamos -1 por 0 para que no afecte numéricamente
    return np.column_stack((no_previo, dias_validos))


pdays_transformer = FunctionTransformer(transform_pdays, validate=False)




#pipeline para las categóricas con nulos: Imputación + OneHotEncoding
cat_nulos_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

#pipeline para el resto , solo el OneHotEncoding
cat_pipe = Pipeline(steps=[
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')) #por si hubiese algun valor nulo
])


#unimos todo en ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipe, num_cols),
        ('cat_nulos', cat_nulos_pipe, cat_cols_con_nulos),
        ('cat_normales', cat_pipe, cat_cols),
        ('pdays_custom', pdays_transformer, pdays_col)
    ],
    remainder='passthrough' 
)   

In [ ]:
import time
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

#convertimos la variable objetivo a números 
y_train_num = y_train.map({'no': 0, 'yes': 1})
y_test_num = y_test.map({'no': 0, 'yes': 1})

#usamos StandardScaler como punto de partida para el escalado
knn_default_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier()) 
])

#medimos tiempo de entrenamiento directo
print("Entrenando KNN con hiperparámetros por defecto...")
inicio = time.time()
knn_default_pipe.fit(X_train, y_train_num)
final = time.time()
tiempo_default = final - inicio

#evaluamos usando la validación cruzada 
cv_scores_default = cross_val_score(knn_default_pipe, X_train, y_train_num, cv=cv_inner, scoring=scoring_metric, n_jobs=-1)

print(f"Tiempo de entrenamiento (Default): {tiempo_default:.4f} segundos")
print(f"Puntuación media ({scoring_metric}) en validación interna: {cv_scores_default.mean():.4f}")

Entrenando KNN con hiperparámetros por defecto...
Tiempo de entrenamiento (Default): 0.0877 segundos
Puntuación media (roc_auc) en validación interna: 0.8133


In [ ]:
from sklearn.preprocessing import  MinMaxScaler, RobustScaler
from sklearn.model_selection import GridSearchCV

#creamos el pipeline
knn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

#definimos la cuadricula de parametros
param_grid_knn = {
    'scaler': [StandardScaler(), MinMaxScaler(), RobustScaler()],
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35]
}

#configuramos la busqueda
grid_knn = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid_knn,
    scoring=scoring_metric, #roc_auc
    cv=cv_inner,            
    n_jobs=-1               
)

print("Entrenando y evaluando modelos KNN... espera un momento.")
#inicimaos cronómetro
inicio = time.time()
grid_knn.fit(X_train, y_train_num)
#finalizamos cronómetro
final = time.time()
tiempo = final - inicio
print("\nEntrenamiento completado")
print(f"El tiempo de ejucción del Gridsearch es {tiempo:.4f} segundos")
print(f"Mejor combinación encontrada: {grid_knn.best_params_}")
print(f"Mejor puntuación ({scoring_metric}) en validación interna: {grid_knn.best_score_:.4f}")

Entrenando y evaluando modelos KNN... espera un momento.

¡Entrenamiento completado!
El tiempo de ejucción del Gridsearch es 19.2540 segundos
Mejor combinación encontrada: {'knn__n_neighbors': 25, 'scaler': RobustScaler()}
Mejor puntuación (roc_auc) en validación interna: 0.8791
